# Classical additive decomposition (manual implementation)

This notebook shows how to manually perform additive decomposition.

Model:
$$Y_t = T_t + S_t + R_t$$


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
def calculate_trend(series, window):
    series = pd.Series(series).reset_index(drop=True)
    n = len(series)
    trend = [np.nan] * n

    # Odd window -> MA
    if window % 2 == 1:

        half = (window - 1) // 2

        for i in range(n):

            if i < half or i >= n - half:
                continue

            trend[i] = series.iloc[
                i - half : i + half + 1
            ].mean()

    # Even window -> CMA
    else:

        ma = []

        for i in range(n - window + 1):

            ma.append(
                series.iloc[i : i + window].mean()
            )

        for i in range(len(ma) - 1):

            cma = (ma[i] + ma[i + 1]) / 2

            trend_index = i + window // 2

            trend[trend_index] = cma

    return trend


def additive_decomposition(df, value_col, period, window):

    result = df.copy()
    series = result[value_col].reset_index(drop=True)
    n = len(series)

    # Trend
    result['Trend'] = calculate_trend(series, window)

    # Remove trend
    result['Y_minus_T'] = (
        result[value_col] - result['Trend']
    )

    # Season number
    result['Season'] = [i % period for i in range(n)]

    # Seasonal effects
    seasonal = {}

    for season in range(period):

        values = result.loc[
            result['Season'] == season,
            'Y_minus_T'
        ].dropna()

        seasonal[season] = values.mean()

    # Normalize
    avg = np.mean(list(seasonal.values()))

    seasonal_norm = {}

    for season in seasonal:

        seasonal_norm[season] = (
            seasonal[season] - avg
        )

    # Assign seasonality
    result['Seasonality'] = [
        seasonal_norm[s]
        for s in result['Season']
    ]

    # Residuals
    result['Residual'] = (
        result[value_col]
        - result['Trend']
        - result['Seasonality']
    )

    return result


## Example 1 - odd window (MA)

In [3]:
df1 = pd.DataFrame({
    'Sales': [
        100,108,117,
        106,114,123,
        112,120,129,
        118,126,135,
        124,132,141
    ]
})

result1 = additive_decomposition(
    df1,
    value_col='Sales',
    period=3,
    window=3
)

result1

,Sales,Trend,Y_minus_T,Season,Seasonality,Residual
0,100,NaN,NaN,0,-6.333333,NaN
1,108,108.333333,-0.333333,1,-0.333333,4.329870e-15
2,117,110.333333,6.666667,2,6.666667,5.329071e-15
3,106,112.333333,-6.333333,0,-6.333333,5.329071e-15
4,114,114.333333,-0.333333,1,-0.333333,4.329870e-15
5,123,116.333333,6.666667,2,6.666667,5.329071e-15
6,112,118.333333,-6.333333,0,-6.333333,5.329071e-15
7,120,120.333333,-0.333333,1,-0.333333,4.329870e-15
8,129,122.333333,6.666667,2,6.666667,5.329071e-15
9,118,124.333333,-6.333333,0,-6.333333,5.329071e-15


## Example 2 - even window (CMA)

In [4]:
df2 = pd.DataFrame({
    'Sales': [
        95,105,120,110,
        115,125,140,130,
        135,145,160,150
    ]
})

result2 = additive_decomposition(
    df2,
    value_col='Sales',
    period=4,
    window=4
)

result2

,Sales,Trend,Y_minus_T,Season,Seasonality,Residual
0,95,NaN,NaN,0,-5.0,NaN
1,105,NaN,NaN,1,0.0,NaN
2,120,110.0,10.0,2,10.0,0.0
3,110,115.0,-5.0,3,-5.0,0.0
4,115,120.0,-5.0,0,-5.0,0.0
5,125,125.0,0.0,1,0.0,0.0
6,140,130.0,10.0,2,10.0,0.0
7,130,135.0,-5.0,3,-5.0,0.0
8,135,140.0,-5.0,0,-5.0,0.0
9,145,145.0,0.0,1,0.0,0.0
